# Multi-Label Chest X-Ray Abnormality Detection & Explainable AI
### Exploratory Data Analysis, Model Inspection, and Grad-CAM Explanations

This notebook explores the NIH ChestX-ray14 dataset, verifies the patient-stratified split, inspects label distributions, and demonstrates target-specific Grad-CAM explainability.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
PROCESSED_CSV = DATA_DIR / "processed" / "labels.csv"
SELECTED_LABELS_JSON = DATA_DIR / "selected_labels.json"
MANIFEST_JSON = DATA_DIR / "chestxray14" / "dataset_manifest.json"

# Load selected labels
with open(SELECTED_LABELS_JSON, "r") as f:
    selected_labels = json.load(f)
print("Selected abnormalities:", selected_labels)

## 1. Dataset Manifest & Integrity Summary

In [ ]:
with open(MANIFEST_JSON, "r") as f:
    manifest = json.load(f)

print("Dataset Status:", manifest["status"])
print("Metadata Rows:", manifest["metadata_rows"])
print("Available Images:", manifest["available_local_images"])
print("Corrupted Images:", manifest["corrupted_images_detected"])

## 2. Split Analysis & Patient Leakage Check

In [ ]:
df = pd.read_csv(PROCESSED_CSV)
print("Total dataset subset size:", len(df))
print("Split value counts:")
print(df["split"].value_counts())

# Check patient overlap
train_p = set(df[df["split"] == "train"]["Patient ID"])
val_p = set(df[df["split"] == "val"]["Patient ID"])
test_p = set(df[df["split"] == "test"]["Patient ID"])

print(f"Patient overlap Train-Val: {len(train_p.intersection(val_p))}")
print(f"Patient overlap Train-Test: {len(train_p.intersection(test_p))}")
print(f"Patient overlap Val-Test: {len(val_p.intersection(test_p))}")

## 3. Label Co-occurrence & Correlation Matrix

In [ ]:
label_corr = df[selected_labels].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(label_corr, annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Abnormality Correlation Heatmap", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Visualizing Sample Chest Radiographs

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, (_, row) in enumerate(df.head(6).iterrows()):
    fname = Path(row["image_path"]).name
    img_path = DATA_DIR / "chestxray14" / "images" / fname
    img = Image.open(img_path)
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"{fname}\n{row['Finding Labels']}", fontsize=9)
    axes[i].axis("off")

plt.tight_layout()
plt.show()